# OPENING NOTES
The core focus of this experimentation notebook is improving the generalisation of the RAF-DB images. The key problems I noticed with my previous implementation were:
- RAF-DB and FER2013's test sets were using the same data augmentations. This is troublesome as RAF-DB is another in-the-wild dataset with its own resolution and dataset-independent format. I need to analyse why exactly the translation to this unseen dataset is especially difficult
- Uncertainty is not being quantified, which is needed in foreign data.
- Timm's `SwinTransformerBlock` is currently a black box to me. While I am evidently replacing the only part of it that allows for Swin-Xception to take effect (replacing the MLP head with a DS-FFN), I want to see how it works internally, as to then see if any parts need my adjustment, as well as further enhance my learning.

# KEY CHANGES
- Examine key errors when classifying RAF-DB [ ]
- Predict with uncertainty estimation using MCDO [x]
- Implement a Swin block manually [x]
- Swin-Base embedding and head quantity [x]

# 0. Import all core dependencies

In [3]:
# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision import models

# Optimisers and Schedulers
from torch.optim import AdamW, RMSprop, SGD
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR, OneCycleLR

# Data Handling
from PIL import Image
import numpy as np
import pandas as pd
import cv2

# Data Visualisation - TODO
import matplotlib.pyplot as plt
import seaborn as sns

# Class Imbalance Handling - TODO
from imblearn.over_sampling import SMOTE
from collections import Counter

# Evaluation - TODO
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    recall_score
)

# Explainability - TODO
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Utilities
from tqdm import tqdm
from einops import rearrange
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
import random
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Configure PyTorch to use my GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

Using cuda


# 1. Define all modules in Swin-Xception

## 1.1 Patch Embedding Block

In [4]:
class PatchEmbedding(nn.Module):
    def __init__(self,
                 in_channels:int=3,
                 dim:int=128,
                 patch_size:int=4):
        super(PatchEmbedding, self).__init__()

        self.patch_size = patch_size
        self.dim = dim

        self.patcher = nn.Conv2d(in_channels=in_channels, out_channels=dim, kernel_size=patch_size, stride=patch_size, padding=0)

        self.flatten = nn.Flatten(start_dim=2, end_dim=3)

        self.norm = nn.LayerNorm(dim)

    def forward(self, x):

        image_resolution = x.shape[-1]

        assert image_resolution % self.patch_size == 0, \
            f"Input image size must be divisible by patch size, \
            image shape: {image_resolution}, patch size: {self.patch_size}"
        
        x_patched = self.patcher(x)

        x_flattened = self.flatten(x_patched)

        x = x_flattened.permute(0,2,1)

        x = self.norm(x)

        return x

## 1.2 Patch Merging Block

In [5]:
class PatchMerging(nn.Module):
    def __init__(self, dim):
        super(PatchMerging, self).__init__()

        self.dim = dim

        self.reduction = nn.Linear(4*dim, 2*dim, bias=False)

        self.norm = nn.LayerNorm(4*dim)

    def forward(self, x):

        B, N, C = x.shape

        H = W = int(np.sqrt(N))

        assert H % 2 == 0 and W % 2 == 0, f"H and W must be even, got H={H}, W={W}"

        x = x.view(B, H, W, C)

        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]

        x = torch.cat([x0,x1,x2,x3], dim=-1)

        x = x.view(B, -1, 4*C)

        x = self.norm(x)

        x = self.reduction(x)

        return x

## 1.3 Depthwise Separable Convolution (The Xception side)

In [6]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self,
                 in_channels,
                 out_channels,
                 kernel_size:int=3):
        super(DepthwiseSeparableConv, self).__init__()

        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, padding=kernel_size//2, groups=in_channels)
        
        self.gelu = nn.GELU()

        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):

        x = self.depthwise(x)

        x = self.gelu(x)

        x = self.pointwise(x)

        return x
    

## 1.4 Depthwise Separable FFN (Replaces MLP Head of Swin Block)

In [7]:
class DepthwiseSeparableFFN(nn.Module):
    def __init__(self,
                 dim,
                 mlp_ratio:int=4,
                 dropout:float=0.5):
        super(DepthwiseSeparableFFN, self).__init__()

        hidden_dim = int(dim * mlp_ratio)

        self.depthwise1 = DepthwiseSeparableConv(dim, hidden_dim, kernel_size=3)

        self.dropout1 = nn.Dropout(dropout)

        self.depthwise2 = DepthwiseSeparableConv(hidden_dim, dim, kernel_size=3)

        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):

        B, N, C = x.shape

        H = W = int(np.sqrt(N))

        x = x.transpose(1, 2).reshape(B, C, H, W)

        x = self.depthwise1(x)

        x = self.dropout1(x)

        x = self.depthwise2(x)

        x = self.dropout2(x)

        x = x.reshape(B, C, N).transpose(1, 2)

        return x

## 1.5 Swin-Xception Block

### 1.5.1 Multi-Head Self Attention Mechanism

In [8]:
class ShiftedWindowMSA(nn.Module):
    def __init__(self,
                 dim,
                 num_heads,
                 window_size:int=7,
                 shifted:bool=True):
        super(ShiftedWindowMSA, self).__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shifted = shifted

        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Query Key Value projection and output
        
        self.linear1 = nn.Linear(dim, 3*dim)
        self.linear2 = nn.Linear(dim, dim)

        # Relative position embeddings
        # Matrix size must be (2M - 1, 2M -1) where M is window size
        
        self.pos_embeddings = nn.Parameter(torch.randn(window_size*2 - 1, window_size*2 - 1))
        
        # Precompute relative indices (window_size^2, window_size^2)
        
        coords = torch.stack(
            torch.meshgrid(
                torch.arange(window_size),
                torch.arange(window_size),
                indexing="ij"
            ),
            dim=-1
        ) # (window_size, window_size, 2)

        coords_flat = coords.view(-1, 2) # (window_size^2, 2)

        rel = coords_flat[:, None, :] - coords_flat[None, :, :] # (window_size^2, window_size^2, 2)

        rel += window_size -1 # shift to be >= 0

        self.register_buffer("relative_indices", rel, persistent=False)

        # Attention mask for shifted windows

        if self.shifted:
            Ws2 = window_size**2
            
            mask = torch.zeros(Ws2, Ws2)
            half = window_size // 2

            mask[-window_size * half:, 0:-window_size * half] = float("-inf")
            mask[0:-window_size * half, -window_size * half:] = float("-inf")

            # reshape for broadcasting: (1,1,1,1,Ws^2,Ws^2)

            mask = mask.view(1, 1, 1, 1, Ws2, Ws2)
            
            # store as buffer

            self.register_buffer("shift_mask", mask, persistent=False)
        else:
            self.register_buffer("shift_mask", None, persistent=False)

    def forward(self, x):
        B, N, C = x.shape
        H = W = int(np.sqrt(N))
        assert H * W == N, "Input sequence must be a perfect square"

        # QKV projection

        qkv = self.linear1(x) # [B, N, 3C]


        # [B, H*W, 3C] -> [B, H, W, 3C]

        qkv = rearrange(qkv, "b (h w) c -> b h w c", h=H, w=W)

        # optional shift

        if self.shifted:
            shift = self.window_size // 2
            qkv = torch.roll(qkv, shifts=(-shift, -shift), dims=(1, 2))

        #partition into windows

        # [B, H, W, 3C] -> [B, Wh, Ww, Ws*Ws, 3C]
        # Wh is window_height
        # Ww is window_width
        # Ws is window_size

        qkv = rearrange(qkv, "b (Wh w1) (Ww w2) c -> b Wh Ww (w1 w2) c", w1=self.window_size, w2=self.window_size)

        # split heads and QKV

        # [B, Wh, Ww, Ws^2, 3C] -> [B, Wh, Ww, Ws^2, 3, num_heads, head_dim]

        qkv = qkv.view(B, qkv.shape[1], qkv.shape[2], qkv.shape[3], 3, self.num_heads, self.head_dim)

        Q, K, V = qkv.unbind(dim=4) # each: [B, Wh, Ww, Ws^2, num_heads, head_dim]


        # move heads forward: [B, num_heads, Wh, Ww, Ws^2, head_dim]
        
        Q = Q.permute(0, 4, 1, 2, 3, 5)
        K = K.permute(0, 4, 1, 2, 3, 5)
        V = V.permute(0, 4, 1, 2, 3, 5)

        # attention: [B, num_heads, Wh, Ww, Ws^2, Ws^2]

        attention = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        # relative position bias: [Ws^2, Ws^2]

        rel_pos = self.pos_embeddings[
            self.relative_indices[..., 0],
            self.relative_indices[..., 1]
        ]
        attention = attention + rel_pos # broadcast over batch, heads, windows

        # shifted window masks

        if self.shifted:
            attention = attention + self.shift_mask

        attention = F.softmax(attention, dim=-1)

        # output: [B, num_heads, Wh, Ww, Ws^2, head_dim]

        out = torch.matmul(attention, V)

        # merge heads: [B, Wh, Ww, Ws^2, num_heads * head_dim]

        out = out.permute(0, 2, 3, 4, 1, 5).contiguous()
        out = out.view(B, out.shape[1], out.shape[2], out.shape[3], C)

        # reverse window partition

        out = rearrange(out, "b Wh Ww (w1 w2) c -> b (Wh w1) (Ww w2) c", w1=self.window_size, w2=self.window_size)

        # reverse shift

        if self.shifted:
            shift = self.window_size // 2
            out = torch.roll(out, shifts=(shift, shift), dims=(1, 2))

        # [B, H, W, C] -> [B, N, C]

        out = rearrange(out, "b h w c -> b (h w) c")
        
        return self.linear2(out)

### 1.5.2 Swin Encoder

In [9]:
class SwinEncoder(nn.Module):
    def __init__(self, dim, num_heads, window_size=7, shift_size=0):
        super(SwinEncoder, self).__init__()
        self.shift_size = shift_size

        self.attention = ShiftedWindowMSA(
            dim=dim,
            num_heads=num_heads,
            window_size=window_size,
            shifted=(shift_size>0)
        )
        
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

        # Instead of an MLP Head, I use my Depthwise Separable FeedForward Network
        self.DSFFN = DepthwiseSeparableFFN(dim=dim)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.DSFFN(self.norm2(x))

        return x

## 1.6 Swin-Xception Backbone

In [10]:
class SwinXception(nn.Module):
    def __init__(self, num_classes:int=7, dropout:float=0.5):
        super(SwinXception, self).__init__()

        self.patch_embed = PatchEmbedding(in_channels=3, dim=128, patch_size=4)

        self.layer1 = nn.ModuleList([SwinEncoder(128, num_heads=4, shift_size=0 if i % 2 == 0 else 3) for i in range(2)])
        self.merge1 = PatchMerging(128) # Reduce dimensions, increase channels

        self.layer2 = nn.ModuleList([SwinEncoder(256, num_heads=8, shift_size=0 if i % 2 == 0 else 3) for i in range(2)])
        self.merge2 = PatchMerging(256)

        self.layer3 = nn.ModuleList([SwinEncoder(512, num_heads=16, shift_size=0 if i % 2 == 0 else 3) for i in range(18)])
        self.merge3 = PatchMerging(512)

        self.layer4 = nn.ModuleList([SwinEncoder(1024, num_heads=32, shift_size=0 if i % 2 == 0 else 3) for i in range(2)])

        self.norm = nn.LayerNorm(1024)
        
        self.avgpool1d = nn.AdaptiveAvgPool1d(output_size=1)

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):

        x = self.patch_embed(x)

        for block in self.layer1:
            x = block(x)
        x = self.merge1(x)

        for block in self.layer2:
            x = block(x)
        x = self.merge2(x)

        for block in self.layer3:
            x = block(x)
        x = self.merge3(x)

        for block in self.layer4:
            x = block(x)

        x = self.norm(x)
        x = x.transpose(1, 2)
        x = self.avgpool1d(x)
        x = torch.flatten(x, 1)
        x = self.head(x)

        return x

    def monte_carlo_dropout_predict(self, x, n_samples=10):
        self.train()
        predictions = []

        with torch.no_grad():
            for _ in range(n_samples):
                predictions.append(torch.softmax(self(x), dim=1))

        predictions = torch.stack(predictions)
        mean_pred = predictions.mean(dim=0)
        uncertainty = predictions.std(dim=0)

        return mean_pred, uncertainty

# 2. Image Preprocessing function

In [11]:
NOTEBOOK_DIR = os.getcwd()

SRC_PATH = os.path.abspath("../../src")

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

from datasets import FERDataset
from torchvision.transforms import v2

transform_train = v2.Compose([
    v2.Resize(size=(224, 224), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness = 0.2, contrast = 0.2),
    v2.RandomRotation(10),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test_fer = v2.Compose([
    v2.RandomResizedCrop(size=(224, 224), antialias=True),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test_raf = v2.Compose([
    v2.RandomResizedCrop(size=(224, 224), antialias=True),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


fer_train = FERDataset(os.path.abspath("../../datasets/FER2013/train"), transform_train)
fer_test = FERDataset(os.path.abspath("../../datasets/FER2013/test"), transform_test_fer)
raf_test = FERDataset(os.path.abspath("../../datasets/RAFDB/DATASET/test"), transform_test_raf)

train_loader = DataLoader(fer_train, batch_size=32, shuffle=True, num_workers=4)
test_fer_loader = DataLoader(fer_test, batch_size=32, shuffle=False, num_workers=4)
test_raf_loader = DataLoader(raf_test, batch_size=32,shuffle=False, num_workers=4)

print(f"FER2013 Training set images: {len(fer_train)}")
print(f"FER2013 Test set images: {len(fer_test)}")
print(f"RAF-DB Test set images: {len(raf_test)}")

FER2013 Training set images: 28709
FER2013 Test set images: 7178
RAF-DB Test set images: 3068


# 3. Stage 1: End-to-End Training

## 3.1 Train Function

In [12]:
def train_one_epoch(model, data_loader, criterion, optimiser, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(data_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimiser.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        # The use of these functions signify backpropagation and changing the lr and weight of my optimiser, hence only used in training
        loss.backward()
        optimiser.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(data_loader)
    epoch_acc = correct * 100. / total
    return epoch_loss, epoch_acc

## 3.2 Evaluation Function

In [13]:
def validate(model, data_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(data_loader):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss = running_loss / len(data_loader)
    val_acc = correct * 100. / total
    
    return val_loss, val_acc

## 3.3 Main Epoch Loop

In [14]:
PATH = "model_checkpoints/latest.pth"
os.makedirs("model_checkpoints", exist_ok=True)

epochs = 50

model = SwinXception(num_classes=7).to(device)

optimiser = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)

start_epoch = 0

if os.path.exists(PATH):
    checkpoint = torch.load(PATH, map_location=device)
    start_epoch = checkpoint["epoch"] + 1
    print(f"Checkpoint found! Starting from epoch {start_epoch}...")
    
    model.load_state_dict(checkpoint["model_state_dict"])
    optimiser.load_state_dict(checkpoint["optimiser_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
else:
    print("No model checkpoints found. Starting from epoch 1...")

for epoch in range(start_epoch, epochs):
    print("="*60)
    print(f"Epoch {epoch+1}/{epochs}")
    
    print("Training on FER2013...")
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimiser, device)

    print("Testing on FER2013...")
    fer_loss, fer_acc = validate(model, test_fer_loader, criterion, device)

    print("Testing of RAF-DB...")
    raf_loss, raf_acc = validate(model, test_raf_loader, criterion, device)
    
    scheduler.step()

    print(f"FER2013 Training Loss: {train_loss:.4f}   Accuracy: {train_acc}")
    print(f"FER2013 Testing  Loss: {fer_loss:.4f}   Accuracy: {fer_acc}")
    print(f"RAF-DB Testing   Loss: {raf_loss:.4f}   Accuracy: {raf_acc}")

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimiser_state_dict": optimiser.state_dict(),
        "scheduler_state_dict": scheduler.state_dict()
    }

    torch.save(checkpoint, PATH)
    
print("="*60)

No model checkpoints found. Starting from epoch 1...
Epoch 1/50
Training on FER2013...


  1%|▋                                                                               | 8/898 [02:04<3:50:50, 15.56s/it]


KeyboardInterrupt: 

# 4. Stage 2: Feature Extraction and Synthesis

## 4.1 Extract Features

In [10]:
def extract_features(model, dataloader, device):
    model.eval()
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader):
            images = images.to(device)

            x = model.patch_embed(images)

            for block in model.layer1:
                x = block(images)
            x = merge1(images)

            for block in model.layer2:
                x = block(images)
            x = merge2(images)

            for block in model.layer3:
                x = block(images)
            x = merge3(images)
            
            for block in model.layer4:
                x = block(images)

            x = model.norm(x)
            x = x.transpose(1, 2)
            x = model.avgpool1d(x)
            x = torch.flatten(x, 1)

            all_features.append(x.cpu().numpy())
            all_labels.append(labels.numpy())

        features = np.vstack(all_features)
        labels = np.concatenate(all_labels)

        return features, labels

## 4.2 Apply SMOTE on extracted deep features

In [12]:
def apply_smote(features, labels, random_state=42):
    print("Original class distribution:")
    print(Counter(labels))
    
    smote = SMOTE(random_state=random_state, k_neighbors=5)
    balanced_features, balanced_labels = smote.fit_resample(features, labels)

    print("Balanced class distribution:")
    print(Counter(balanced_labels))

    return balanced_features, balanced_labels

In [ ]:
features, labels = extract_features(model, train_loader, device)
balanced_features, balanced_labels = apply_smote(features, labels)

# 5. Stage 3: Retraining the MLP Head

In [15]:
def retrain_mlp_head(model, features, labels, device, epochs=20, batch_size=128, lr=1e-3, weight_decay=1e-3):
    # Freeze all params in Swin-X backbone
    for param in model.patch_embed.parameters():
        param.requires_grad = False
    for stage in [model.layer1, model.layer2, model.layer3, model.layer4]:
        for block in stage:
            for param in block.parameters():
                param.requires_grad = False

    for merge in [model.merge1, model.merge2, model.merge3, model.merge4]:
        for param in merge.parameters():
            param.requires_grad = False

    features_tensor = torch.FloatTensor(features)
    labels_tensor = torch.LongTensor(features)
    dataset = torch.utils.data.TensorDataset(features_tensor, labels_tensor)
    dataloader = Dataloader(dataset, batch_size=batch_size, shuffle=True)

    criterion = nn.CrossEntropyLoss()
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for features, labels in tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}"):
            features = features.to(device)
            labels = labels.to(device)

            optimiser.zero_grad()

            outputs = model.head(features)

            loss = criterion(outputs, labels)

            loss.backward()
            optimiser.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(data_loader)
        epoch_acc = correct * 100. / total

        print(f"Epoch {epoch+1}/{epochs}   Loss: {epoch_loss}   Accuracy: {epoch_acc}")

    return model

In [ ]:
model = retrain_mlp_head(model, balanced_features, balanced_labels, device)

torch.save(model.state_dict(), 'swin_xception_final.pth')

# 6. Evaluation Metrics and Visualisation

## 6.1 Validate Loss and Accuracy on Test sets

In [ ]:
fer_loss, fer_acc = validate(model, test_fer_loader, criterion, device)
raf_loss, raf_acc = validate(model, test_raf_loader, criterion, device)

print(f"FER2013 new loss: {fer_loss:.4f}   new accuracy: {fer_acc:.2f}")
print(f"RAF- new loss: {fer_loss:.4f}   new accuracy: {fer_acc:.2f}")

# Experimentation 2 - Outcomes:
- timm's SwinTransformerBlock is rigorously optimised to work with my hardware specifications
- Building multi-head self attention from scratch introduces gnarly overhead
- No further alterations can be made to the data augments on RAF-DB test
- `model.compile()` is strictly for Linux, as Triton is strictly for Linux
- I can still swap components of timm's block as I have done with the DS-FFN
- Data-dependent indexing in a compiled graph causes slowdown; when trying to apply individual row and column masks for shifted window MSA, matrix multiplication may operate faster as there are less rows and columns, but mask updates and softmax run slower due to the dynamic sizes being multiplied.